# 01 — Gateway: Lambda e MCP no Runtime

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a seleção é declarada no deploy. A jornada `single-account` inclui 00–09 e 15–16; Registry, Terraform e RAG são exigidos somente pelas aulas que os utilizam. Ampliar a seleção exige atualizar a implantação pelo IaC. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 01](../docs/diagramas/svg/01_gateway_targets.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "01")
print({"lab": "01", "dependencies": list(dependencies), "region": config["region"]})

### Conectar ao ambiente implantado

Lemos os outputs exatos do CloudFormation. O cliente não cria infraestrutura e não substitui um serviço indisponível.

In [ ]:
lab = AWSWorkshop(CONFIG, domain="platform")
print(lab.preflight())

### Autenticar a pessoa no Cognito

O login usa USER_PASSWORD_AUTH para usuários sintéticos do workshop e SECRET_HASH para o cliente confidencial. Em uma aplicação de usuários finais, use authorization code com PKCE. O código abaixo mostra a chamada Cognito e a verificação de assinatura, emissor, audiência e token_use. Só imprimimos claims selecionadas.

In [ ]:
from santander_demo.aws_workshop.identity import login, verify
display(Code(inspect.getsource(login), language="python"))
display(Code(inspect.getsource(verify), language="python"))
token = lab.login("c", "supervisor")
claims = lab.claims(token)
print({key: claims[key] for key in ["iss", "sub", "aud", "token_use"]})

### Executar tools/list no Gateway

Abra uma sessão MCP sobre HTTPS. initialize negocia o protocolo; tools/list devolve os schemas dos targets. Os prefixos qualificam os nomes que aparecem nas ações Cedar.

In [ ]:
from santander_demo.aws_workshop.client import gateway_session, content
async with gateway_session(lab.gateway["GatewayUrl"], token) as mcp:
    announced = await mcp.list_tools()
    for tool in announced.tools:
        print(tool.name, tool.inputSchema)
assert "riscos___consultar_risco" in {t.name for t in announced.tools}

### Chamar o target Lambda

O JWT é validado na entrada. Após o permit da Policy, o Gateway assina a invocação Lambda com sua role IAM. O backend valida a entrada e retorna dados sintéticos.

In [ ]:
async with gateway_session(lab.gateway["GatewayUrl"], token) as mcp:
    response = await mcp.call_tool("riscos___consultar_risco", {
        "cliente_id": "C001", "max_registros": 1, "request_id": str(uuid.uuid4())
    })
risk = content(response)
assert risk["customer_id"] == "C001" and risk["synthetic"] is True
print(risk)

### Chamar o servidor MCP no Runtime

O segundo target é um servidor MCP hospedado no AgentCore Runtime. O Gateway assina a chamada com IAM/SigV4. O interceptor encaminha o JWT verificado da pessoa em um header separado; o backend valida assinatura/audiência antes de ler sua ACL independente.

In [ ]:
fraud = await lab.call(token, "especialista___consultar_fraude",
                       {"cliente_id": "C001", "max_registros": 1})
assert fraud["result"]["source_kind"] == "AWS_SIMULATED_DATABRICKS"
assert fraud["result"]["verified_subject"] == claims["sub"]
print(fraud)

### Inspecionar IAM, JWT humano e a restrição de entrada

A API não aceita JWT_PASSTHROUGH para esse tipo de target. Usamos GATEWAY_IAM_ROLE com assinatura bedrock-agentcore. O JWT segue em um header próprio, sobrescrito pelo interceptor. A resource policy do Runtime nega qualquer role diferente da role do Gateway. AllowedWorkloadConfiguration exige transaction tokens do target nativo HTTP de Runtime e não se aplica a esta agregação MCP.

In [ ]:
from santander_demo.aws_workshop.interceptor import transform
from santander_demo.aws_workshop.backend import verified_source_claims
from santander_demo.aws_workshop.runtime_boundary import policy
display(Code(inspect.getsource(transform), language="python"))
display(Code(inspect.getsource(verified_source_claims), language="python"))
display(Code(json.dumps(policy(lab.backend["RuntimeArn"], lab.gateway["GatewayRoleArn"]),
                        indent=2), language="json"))

**Aceite:** ferramentas descobertas pelo MCP, resposta Lambda válida e resposta do Runtime ligada ao `sub` autenticado. As três provas vêm da AWS.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.